In [3]:
"""Causal Kalman innovation reversal factor for BigAlpha 2026.

The local-linear Kalman model predicts today's log price using only states from
previous observations. The factor takes the opposite side of the standardized
one-step innovation, then updates the state with today's close. This ordering is
important: the signal never uses a smoothed state that already contains today's
observation.

The process and observation variances adapt online using only past returns. New
or re-entering constituents use the platform's five-day reversal for warm-up.
The final signal is ranked, industry demeaned, and standardized by trading day.
"""

import numpy as np
import pandas as pd


def _clean_result(df):
    columns = ["date", "instrument", "factor"]
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=columns)
    df = df[columns].copy()
    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["factor"] = pd.to_numeric(df["factor"], errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan)
    finite_factor = np.isfinite(df["factor"].to_numpy(dtype=float))
    df = df.loc[finite_factor].dropna(subset=["date", "instrument"])
    df = df.groupby(
        ["date", "instrument"], as_index=False, observed=True
    )["factor"].mean()
    return df.sort_values(["date", "instrument"]).reset_index(drop=True)


def _kalman_group(log_prices, dates):
    """Return causal negative standardized innovations for one instrument."""
    n = len(log_prices)
    signal = np.full(n, np.nan, dtype=float)
    if n == 0:
        return signal

    initialized = False
    level = 0.0
    trend = 0.0
    p00 = 0.0
    p01 = 0.0
    p11 = 0.0
    obs_var = 1.0e-4
    previous_price = np.nan
    previous_date = None
    observations = 0

    for i in range(n):
        price = log_prices[i]
        current_date = dates[i]
        if not np.isfinite(price):
            continue

        gap_days = None
        if previous_date is not None:
            gap_days = (current_date - previous_date).days

        # A long membership gap means the old state is no longer reliable.
        if (not initialized) or (gap_days is not None and gap_days > 45):
            level = price
            trend = 0.0
            p00 = 10.0 * obs_var
            p01 = 0.0
            p11 = obs_var
            previous_price = price
            previous_date = current_date
            observations = 1
            initialized = True
            continue

        # Predict with yesterday's state before seeing today's observation.
        level_prediction = level + trend
        trend_prediction = trend
        q_level = max(0.02 * obs_var, 1.0e-8)
        q_trend = max(0.0005 * obs_var, 1.0e-10)
        p00_prediction = p00 + 2.0 * p01 + p11 + q_level
        p01_prediction = p01 + p11
        p11_prediction = p11 + q_trend

        measurement_var = max(obs_var, 1.0e-8)
        innovation = price - level_prediction
        innovation_var = max(p00_prediction + measurement_var, 1.0e-10)

        # Positive price surprise receives a negative reversal signal.
        if observations >= 5:
            signal[i] = -innovation / np.sqrt(innovation_var)

        # Update only after the causal signal has been recorded.
        gain_level = p00_prediction / innovation_var
        gain_trend = p01_prediction / innovation_var
        level = level_prediction + gain_level * innovation
        trend = trend_prediction + gain_trend * innovation
        p00 = max((1.0 - gain_level) * p00_prediction, 1.0e-12)
        p01 = (1.0 - gain_level) * p01_prediction
        p11 = max(p11_prediction - gain_trend * p01_prediction, 1.0e-12)

        # Today's return changes variance only for tomorrow's prediction.
        if np.isfinite(previous_price):
            log_return = price - previous_price
            obs_var = 0.97 * obs_var + 0.03 * log_return * log_return
            obs_var = min(max(obs_var, 1.0e-8), 0.05)

        previous_price = price
        previous_date = current_date
        observations += 1

    return signal


def _calculate_factor(frame, start, end):
    if frame is None or len(frame) == 0:
        return pd.DataFrame(columns=["date", "instrument", "factor"])

    df = frame.copy()
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["close"] = pd.to_numeric(df["close"], errors="coerce")
    df["reversal_5"] = pd.to_numeric(df["reversal_5"], errors="coerce")
    df["industry"] = df["industry"].fillna("UNKNOWN").astype(str)
    df = df.sort_values(["instrument", "date"])
    df = df.drop_duplicates(["date", "instrument"], keep="last").reset_index(drop=True)

    raw_signal = np.full(len(df), np.nan, dtype=float)
    for positions in df.groupby("instrument", sort=False).indices.values():
        positions = np.asarray(positions, dtype=int)
        prices = df.loc[positions, "close"].to_numpy(dtype=float)
        valid_prices = np.where(prices > 0, np.log(prices), np.nan)
        dates = df.loc[positions, "date"].tolist()
        raw_signal[positions] = _kalman_group(valid_prices, dates)

    fallback = df["reversal_5"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    close_tiebreaker = np.log(df["close"].clip(lower=1.0e-8)) * 1.0e-9
    df["cold_start_signal"] = fallback + close_tiebreaker
    df["raw_signal"] = pd.Series(raw_signal, index=df.index).fillna(
        df["cold_start_signal"]
    )

    # Robust daily rank before industry demeaning limits innovation outliers.
    df["rank_signal"] = df.groupby("date")["raw_signal"].rank(
        method="average", pct=True
    )
    industry_mean = df.groupby(["date", "industry"])["rank_signal"].transform("mean")
    df["neutral_signal"] = df["rank_signal"] - industry_mean

    daily_mean = df.groupby("date")["neutral_signal"].transform("mean")
    daily_std = df.groupby("date")["neutral_signal"].transform("std")
    df["factor"] = (df["neutral_signal"] - daily_mean) / daily_std.replace(0.0, np.nan)

    # A defensive fallback prevents a constant cross-section on any warm-up day.
    bad_dates = df.loc[~np.isfinite(df["factor"]), "date"].unique()
    if len(bad_dates) > 0:
        bad_mask = df["date"].isin(bad_dates)
        fallback_rank = df.loc[bad_mask].groupby("date")["cold_start_signal"].rank(
            method="average", pct=True
        )
        fallback_mean = fallback_rank.groupby(df.loc[bad_mask, "date"]).transform("mean")
        fallback_std = fallback_rank.groupby(df.loc[bad_mask, "date"]).transform("std")
        df.loc[bad_mask, "factor"] = (
            fallback_rank - fallback_mean
        ) / fallback_std.replace(0.0, np.nan)

    target = (df["date"] >= start) & (df["date"] <= end)
    return _clean_result(df.loc[target, ["date", "instrument", "factor"]])


def main(data_source=None, start_date=None, end_date=None):
    from bigquant import dai

    if start_date is None:
        start_date = "2019-01-01"
    if end_date is None:
        end_date = "2024-12-31"

    start = pd.to_datetime(start_date).normalize()
    end = pd.to_datetime(end_date).normalize()
    query_start = (start - pd.Timedelta(days=420)).strftime("%Y-%m-%d")
    end_next = (end + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    sql = f"""
    SELECT
        CAST(f.date AS DATE) AS date,
        f.instrument,
        f.close,
        f.reversal_5,
        COALESCE(CAST(e.industry_level1_code AS VARCHAR), 'UNKNOWN') AS industry
    FROM bigalpha_2026_factorlib f
    LEFT JOIN bigalpha_2026_exposure e
      ON CAST(f.date AS DATE) = CAST(e.date AS DATE)
     AND f.instrument = e.instrument
    WHERE f.date >= '{query_start}' AND f.date < '{end_next}'
    ORDER BY f.instrument, f.date
    """

    frame = dai.query(
        sql,
        filters={"date": [query_start, end_next]},
        compression=True,
    ).df()
    calculated = _calculate_factor(frame, start, end)

    # Query the requested-date universe separately. A long lookback query can
    # expose historical instruments that are not in the target day's pool.
    start_day = start.strftime("%Y-%m-%d")
    universe_sql = f"""
    SELECT
        CAST(date AS DATE) AS date,
        instrument
    FROM bigalpha_2026_factorlib
    WHERE date >= '{start_day}' AND date < '{end_next}'
    ORDER BY date, instrument
    """
    universe = dai.query(
        universe_sql,
        filters={"date": [start_day, end_next]},
        compression=True,
    ).df()
    universe["date"] = pd.to_datetime(universe["date"]).dt.normalize()
    universe["instrument"] = universe["instrument"].astype(str)
    universe = universe[["date", "instrument"]].drop_duplicates()

    result = universe.merge(
        calculated,
        on=["date", "instrument"],
        how="left",
        validate="one_to_one",
    )
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce").fillna(0.0)
    daily_mean = result.groupby("date")["factor"].transform("mean")
    daily_std = result.groupby("date")["factor"].transform("std")
    result["factor"] = (result["factor"] - daily_mean) / daily_std.replace(0.0, np.nan)
    return _clean_result(result)
